In [0]:
# Define your project variables
catalog = "capstone_101"

# Verify the catalog exists
print(f"Using catalog: {catalog}")

In [0]:
from pyspark.sql.functions import col, to_timestamp, coalesce, lit, when, split

# 1. Read from Bronze
bronze_data = spark.read.table(f"{catalog}.bronze.raw_sales")

# 2. Build the transformation logic
silver_cleaned = (bronze_data
    .filter(col("UnitPrice").cast("double") > 0)
    
    # CustomerID logic (Keeping our previous fix)
    .withColumn("CustomerID", 
                coalesce(
                    when(col("CustomerID").contains("."), split(col("CustomerID"), r"\.")[0]),
                    col("CustomerID"),
                    lit("Guest")
                )
    )
    
    # FIX: Flexible Date Parsing
    # 'M/d/yyyy H:mm' handles '12/1/2010' and '12/10/2010'
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
)

# 3. Clean and Write
spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.cleaned_sales")

(silver_cleaned.write
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.cleaned_sales"))

# 4. Optimization (Z-ORDER)
spark.sql(f"OPTIMIZE {catalog}.silver.cleaned_sales ZORDER BY (CustomerID)")

print("Silver layer successfully overwritten with fixed Date Parsing!")

In [0]:
display(spark.read.table(f"{catalog}.silver.cleaned_sales").select("InvoiceDate").limit(5))